In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load fact and dimension tables
fact_opp = spark.table('subscription_catalog.gold_schema.fact_opportunities')
dim_date = spark.table('subscription_catalog.gold_schema.dim_date')
dim_customer = spark.table('subscription_catalog.gold_schema.dim_customer')
dim_product = spark.table('subscription_catalog.gold_schema.dim_product')
dim_contract = spark.table('subscription_catalog.gold_schema.dim_contract_type')

# Data overview
print("Data Range:")
fact_opp.select(
    F.min('start_date').alias('earliest_date'),
    F.max('start_date').alias('latest_date'),
    F.count('*').alias('total_records'),
    F.countDistinct('customer_id').alias('unique_customers')
).display()

print("\nClose Status Distribution:")
fact_opp.groupBy('close_status').count().orderBy(F.desc('count')).display()

In [0]:
#1.	How many customers did we gain and lose this period in 2024? 

print("=" * 80)
print("=" * 80)

spark.sql("""
    WITH customers_2023 AS (
        SELECT DISTINCT customer_id
        FROM subscription_catalog.gold_schema.fact_opportunities
        WHERE YEAR(start_date) = 2023
            AND close_status IN ('Active', 'Won')
    ),
    customers_2024 AS (
        SELECT DISTINCT customer_id
        FROM subscription_catalog.gold_schema.fact_opportunities
        WHERE YEAR(start_date) = 2024
            AND close_status IN ('Active', 'Won')
    ),
    lost_customers AS (
        SELECT customer_id
        FROM customers_2023
        WHERE customer_id NOT IN (SELECT customer_id FROM customers_2024)
    ),
    gained_customers AS (
        SELECT customer_id
        FROM customers_2024
        WHERE customer_id NOT IN (SELECT customer_id FROM customers_2023)
    )
    SELECT
        COUNT(DISTINCT customer_id) AS customers_gained,
        (SELECT COUNT(DISTINCT customer_id) FROM lost_customers) AS customers_lost
    FROM gained_customers
""").display()

In [0]:
# KPI 2: Are we losing more customers now compared to previous periods?

print("\n" + "=" * 80)
print("CHURN COMPARISON: Recent vs Previous Quarter")
print("=" * 80)

spark.sql("""
    SELECT 
        CASE 
            WHEN DATE_FORMAT(start_date, 'yyyy-MM') BETWEEN '2024-01' AND '2024-12' 
                THEN 'year'
            -- WHEN DATE_FORMAT(start_date, 'yyyy-MM') BETWEEN '2024-01' AND '2024-03' 
            --     THEN 'Q1 2024 (Jan-Mar)'
        END AS period,
        COUNT(DISTINCT customer_id) AS churned_customers
    FROM subscription_catalog.gold_schema.fact_opportunities
    WHERE close_status = 'Lost'
        AND DATE_FORMAT(start_date, 'yyyy-MM') BETWEEN '2024-01' AND '2024-12'
    GROUP BY period
    ORDER BY period
""").display()

In [0]:
# KPI 3: Who are our highest-value customers by recurring revenue?

print("=" * 80)
print("TOP 20 HIGHEST-VALUE CUSTOMERS")
print("=" * 80)

spark.sql("""
    SELECT 
        customer_id,
        SUM(revenue_in_gbp) AS total_revenue_gbp,
        COUNT(opportunity_id) AS num_contracts,
        COUNT(DISTINCT product_id) AS num_products,
        MIN(start_date) AS customer_since
    FROM subscription_catalog.gold_schema.fact_opportunities
    WHERE close_status IN ('Active', 'Won')
    GROUP BY customer_id
    ORDER BY total_revenue_gbp DESC
    LIMIT 20
""").display()


In [0]:
# KPI 4: Which customers are expanding their product usage over time?print("\n" + "=" * 80)
print("CUSTOMERS EXPANDING (2024 vs 2023)")
print("=" * 80)

spark.sql("""
    WITH customer_yearly AS (
        SELECT 
            customer_id,
            YEAR(start_date) AS year,
            SUM(revenue_in_gbp) AS yearly_revenue
        FROM subscription_catalog.gold_schema.fact_opportunities
        WHERE close_status IN ('Active', 'Won')
            AND YEAR(start_date) IN (2023, 2024)
        GROUP BY customer_id, YEAR(start_date)
    )
    SELECT 
        cy2024.customer_id,
        cy2023.yearly_revenue AS revenue_2023,
        cy2024.yearly_revenue AS revenue_2024,
        cy2024.yearly_revenue - cy2023.yearly_revenue AS revenue_growth,
        ROUND((cy2024.yearly_revenue - cy2023.yearly_revenue) / cy2023.yearly_revenue * 100, 2) AS growth_pct
    FROM customer_yearly cy2024
    JOIN customer_yearly cy2023 ON cy2024.customer_id = cy2023.customer_id
    WHERE cy2024.year = 2024 AND cy2023.year = 2023
        AND cy2024.yearly_revenue > cy2023.yearly_revenue
    ORDER BY revenue_growth DESC
    LIMIT 20
""").display()

In [0]:
# KPI 5: Which customers are reducing or stopping product usage?

print("\n" + "=" * 80)
print("CUSTOMERS CONTRACTING (2024 vs 2023)")
print("=" * 80)

spark.sql("""
    WITH customer_yearly AS (
        SELECT 
            customer_id,
            YEAR(start_date) AS year,
            SUM(revenue_in_gbp) AS yearly_revenue
        FROM subscription_catalog.gold_schema.fact_opportunities
        WHERE close_status IN ('Active', 'Won')
            AND YEAR(start_date) IN (2023, 2024)
        GROUP BY customer_id, YEAR(start_date)
    )
    SELECT 
        cy2024.customer_id,
        cy2023.yearly_revenue AS revenue_2023,
        cy2024.yearly_revenue AS revenue_2024,
        cy2024.yearly_revenue - cy2023.yearly_revenue AS revenue_decline,
        ROUND((cy2024.yearly_revenue - cy2023.yearly_revenue) / cy2023.yearly_revenue * 100, 2) AS decline_pct
    FROM customer_yearly cy2024
    JOIN customer_yearly cy2023 ON cy2024.customer_id = cy2023.customer_id
    WHERE cy2024.year = 2024 AND cy2023.year = 2023
        AND cy2024.yearly_revenue < cy2023.yearly_revenue
    ORDER BY revenue_decline
    LIMIT 20
""").display()

print("\n" + "=" * 80)
print("CUSTOMERS WHO STOPPED (Had 2023 revenue, No 2024 revenue)")
print("=" * 80)

spark.sql("""
    SELECT 
        customer_id,
        SUM(revenue_in_gbp) AS revenue_2023,
        MAX(start_date) AS last_activity_date
    FROM subscription_catalog.gold_schema.fact_opportunities
    WHERE close_status IN ('Active', 'Won')
        AND YEAR(start_date) = 2023
        AND customer_id NOT IN (
            SELECT DISTINCT customer_id 
            FROM subscription_catalog.gold_schema.fact_opportunities
            WHERE YEAR(start_date) = 2024
        )
    GROUP BY customer_id
    ORDER BY revenue_2023 DESC
    LIMIT 20
""").display()

In [0]:
# KPI 6: How much recurring revenue are we retaining from existing customers in current FY?

print("=" * 80)
print("REVENUE RETENTION - FY 2024")
print("=" * 80)

spark.sql("""
    WITH fy2023_base AS (
        SELECT 
            customer_id,
            SUM(revenue_in_gbp) AS revenue_2023
        FROM subscription_catalog.gold_schema.fact_opportunities
        WHERE YEAR(start_date) = 2023 
            AND close_status IN ('Active', 'Won')
        GROUP BY customer_id
    ),
    fy2024_revenue AS (
        SELECT 
            customer_id,
            SUM(revenue_in_gbp) AS revenue_2024
        FROM subscription_catalog.gold_schema.fact_opportunities
        WHERE YEAR(start_date) = 2024 
            AND close_status IN ('Active', 'Won')
        GROUP BY customer_id
    )
    SELECT 
        SUM(b.revenue_2023) AS total_fy2023_revenue,
        SUM(COALESCE(r.revenue_2024, 0)) AS retained_revenue_fy2024,
        COUNT(b.customer_id) AS fy2023_customers,
        COUNT(r.customer_id) AS retained_customers,
        ROUND(SUM(COALESCE(r.revenue_2024, 0)) / SUM(b.revenue_2023) * 100, 2) AS revenue_retention_rate_pct,
        ROUND(COUNT(r.customer_id) / COUNT(b.customer_id) * 100, 2) AS customer_retention_rate_pct
    FROM fy2023_base b
    LEFT JOIN fy2024_revenue r ON b.customer_id = r.customer_id
""").display()



In [0]:
# KPI 7: How much recurring revenue growth is coming from upgrades versus losses in current FY?

print("\n" + "=" * 80)
print("REVENUE GROWTH: UPGRADES VS LOSSES - FY 2024")
print("=" * 80)

spark.sql("""
    WITH fy2023_base AS (
        SELECT 
            customer_id,
            SUM(revenue_in_gbp) AS revenue_2023
        FROM subscription_catalog.gold_schema.fact_opportunities
        WHERE YEAR(start_date) = 2023 
            AND close_status IN ('Active', 'Won')
        GROUP BY customer_id
    ),
    fy2024_revenue AS (
        SELECT 
            customer_id,
            SUM(revenue_in_gbp) AS revenue_2024
        FROM subscription_catalog.gold_schema.fact_opportunities
        WHERE YEAR(start_date) = 2024 
            AND close_status IN ('Active', 'Won')
        GROUP BY customer_id
    )
    SELECT 
        SUM(CASE WHEN r.revenue_2024 > b.revenue_2023 
            THEN r.revenue_2024 - b.revenue_2023 ELSE 0 END) AS revenue_from_upgrades,
        COUNT(CASE WHEN r.revenue_2024 > b.revenue_2023 THEN 1 END) AS customers_upgraded,
        
        SUM(CASE WHEN r.revenue_2024 < b.revenue_2023 
            THEN r.revenue_2024 - b.revenue_2023 ELSE 0 END) AS revenue_from_downgrades,
        COUNT(CASE WHEN r.revenue_2024 < b.revenue_2023 THEN 1 END) AS customers_downgraded,
        
        SUM(CASE WHEN r.revenue_2024 IS NULL 
            THEN b.revenue_2023 ELSE 0 END) AS revenue_lost_to_churn,
        COUNT(CASE WHEN r.revenue_2024 IS NULL THEN 1 END) AS customers_churned
    FROM fy2023_base b
    LEFT JOIN fy2024_revenue r ON b.customer_id = r.customer_id
""").display()

print("\n" + "=" * 80)
print("NEW CUSTOMER REVENUE - FY 2024")
print("=" * 80)

spark.sql("""
    SELECT 
        COUNT(DISTINCT customer_id) AS new_customers,
        SUM(revenue_in_gbp) AS new_customer_revenue
    FROM subscription_catalog.gold_schema.fact_opportunities
    WHERE YEAR(start_date) = 2024 
        AND close_status IN ('Active', 'Won')
        AND customer_id NOT IN (
            SELECT DISTINCT customer_id 
            FROM subscription_catalog.gold_schema.fact_opportunities
            WHERE YEAR(start_date) = 2023
        )
""").display()

In [0]:
# KPI 8: How does revenue year-to-date compare with the same period last year?
# Comparing Jan-Jun 2024 with Jan-Jun 2023

print("=" * 80)
print("YEAR-TO-DATE REVENUE COMPARISON: 2024 vs 2023")
print("=" * 80)

spark.sql("""
    SELECT 
        YEAR(start_date) AS year,
        SUM(revenue_in_gbp) AS total_revenue,
        COUNT(DISTINCT customer_id) AS unique_customers,
        COUNT(opportunity_id) AS total_contracts
    FROM subscription_catalog.gold_schema.fact_opportunities
    WHERE MONTH(start_date) <= 6
        AND YEAR(start_date) IN (2023, 2024)
        AND close_status IN ('Active', 'Won')
    GROUP BY YEAR(start_date)
    ORDER BY year DESC
""").display()

print("\n" + "=" * 80)
print("MONTHLY BREAKDOWN: 2024 vs 2023 (Jan-Jun)")
print("=" * 80)

spark.sql("""
    SELECT 
        MONTH(start_date) AS month,
        YEAR(start_date) AS year,
        SUM(revenue_in_gbp) AS monthly_revenue,
        COUNT(DISTINCT customer_id) AS active_customers
    FROM subscription_catalog.gold_schema.fact_opportunities
    WHERE MONTH(start_date) <= 6
        AND YEAR(start_date) IN (2023, 2024)
        AND close_status IN ('Active', 'Won')
    GROUP BY MONTH(start_date), YEAR(start_date)
    ORDER BY month, year
""").display()

In [0]:
# KPI 9: Which customers or products contribute the most to total recurring revenue?

print("=" * 80)
print("TOP 20 CUSTOMERS BY REVENUE")
print("=" * 80)

spark.sql("""
    SELECT 
        customer_id,
        SUM(revenue_in_gbp) AS total_revenue_gbp,
        COUNT(opportunity_id) AS num_contracts,
        COUNT(DISTINCT product_id) AS num_products,
        MIN(start_date) AS first_purchase,
        MAX(start_date) AS last_purchase
    FROM subscription_catalog.gold_schema.fact_opportunities
    WHERE close_status IN ('Active', 'Won')
    GROUP BY customer_id
    ORDER BY total_revenue_gbp DESC
    LIMIT 20
""").display()

print("\n" + "=" * 80)
print("TOP 20 PRODUCTS BY REVENUE")
print("=" * 80)

spark.sql("""
    SELECT 
        product_id,
        SUM(revenue_in_gbp) AS total_revenue_gbp,
        COUNT(opportunity_id) AS num_sales,
        COUNT(DISTINCT customer_id) AS num_customers,
        ROUND(AVG(revenue_in_gbp), 2) AS avg_revenue_per_sale
    FROM subscription_catalog.gold_schema.fact_opportunities
    WHERE close_status IN ('Active', 'Won')
    GROUP BY product_id
    ORDER BY total_revenue_gbp DESC
    LIMIT 20
""").display()

In [0]:
# KPI 10: What is the rolling 12 month recurring revenue trend?

print("=" * 80)
print("ROLLING 12-MONTH RECURRING REVENUE TREND")
print("=" * 80)

# Monthly revenue
monthly_revenue = spark.sql("""
    SELECT 
        DATE_FORMAT(start_date, 'yyyy-MM') AS year_month,
        SUM(revenue_in_gbp) AS monthly_revenue,
        COUNT(DISTINCT customer_id) AS active_customers
    FROM subscription_catalog.gold_schema.fact_opportunities
    WHERE close_status IN ('Active', 'Won')
    GROUP BY DATE_FORMAT(start_date, 'yyyy-MM')
    ORDER BY year_month
""")

print("\nMonthly Revenue:")
monthly_revenue.display()

# Calculate rolling 12-month revenue
from pyspark.sql.window import Window
from pyspark.sql import functions as F

window_spec = Window.orderBy('year_month').rowsBetween(-11, 0)

rolling_12m = monthly_revenue \
    .withColumn('rolling_12m_revenue', F.sum('monthly_revenue').over(window_spec)) \
    .filter(F.col('year_month') >= '2022-12')

print("\n" + "=" * 80)
print("ROLLING 12-MONTH METRICS")
print("=" * 80)
rolling_12m.select(
    'year_month',
    F.round('rolling_12m_revenue', 2).alias('rolling_12m_revenue_gbp'),
    'monthly_revenue'
).display()

# Simple visualization
import matplotlib.pyplot as plt

trend_df = rolling_12m.toPandas()

plt.figure(figsize=(14, 6))
plt.plot(trend_df['year_month'], trend_df['rolling_12m_revenue'], 
         marker='o', linewidth=2, color='#1f77b4', label='Rolling 12M Revenue')
plt.fill_between(trend_df['year_month'], trend_df['rolling_12m_revenue'], alpha=0.3)
plt.title('Rolling 12-Month Recurring Revenue Trend', fontsize=14, fontweight='bold')
plt.xlabel('Period')
plt.ylabel('Revenue (GBP)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()